## 1 — Functions

A function is a **named, reusable block of code**. Write it once — call it anywhere.

```python
def function_name(parameter1, parameter2):
    """One-line description."""
    # code here
    return result
```

| Keyword | Purpose |
|---------|--------|
| `def` | marks the start of a function definition |
| `( )` | parameters live here |
| `:` | required at the end of the `def` line |
| `return` | sends a value back to the caller |
| indentation | 4 spaces — the function body |


In [ ]:
# Parameters + return value
def add(a, b):
    """Return the sum of a and b."""
    return a + b

print(add(3, 7))    # 10
print(add(10, 20))  # 30


In [ ]:
def route_query(query: str) -> str:
    """Route a customer query to the correct support agent."""
    q = query.lower()
    if   "cancel" in q or "refund"   in q: return "human_agent"
    elif "track"  in q or "where"    in q: return "order_agent"
    elif "return" in q or "exchange" in q: return "returns_agent"
    elif "price"  in q or "discount" in q: return "promotions_agent"
    elif "product" in q or "spec"    in q: return "catalog_agent"
    else:                                   return "general_agent"

queries = [
    "Where is my order?",
    "I want a refund",
    "Do you have discounts?",
]

# ITERATION TRACE:
# Iter 1 → query="Where is my order?"  q="where is my order?"
#           "cancel"? No | "where"? Yes → return "order_agent"
#
# Iter 2 → query="I want a refund"     q="i want a refund"
#           "cancel"? No | "refund"? Yes → return "human_agent"
#
# Iter 3 → query="Do you have discounts?"  q="do you have discounts?"
#           "cancel"? No | "where"? No | "discount"? Yes → return "promotions_agent"

for q in queries:
    print(f"  '{q}'  →  {route_query(q)}")


## 2 — Default Parameter Values

A parameter can have a default value.  
If the caller does not pass it → default is used.  
If the caller does pass it → caller's value wins.

> **Rule:** parameters WITH defaults must come AFTER parameters WITHOUT defaults.
```python
def fn(required, optional=default)   # correct
def fn(optional=default, required)   # SyntaxError
```


In [10]:
def power(base,exponent=2):
    return base ** exponent

power(3,4)

81

In [ ]:
def power(base, exponent=2):
    """Return base raised to exponent. Default: squared."""
    return base ** exponent

print(power(3))       # 3**2 = 9   — uses default exponent=2
print(power(3, 3))    # 3**3 = 27  — caller overrides to 3
print(power(2, 10))   # 2**10 = 1024


---


In [14]:
def total(*numbers):
    """Add any number of values."""
    result = 0
    for n in numbers:
        result += n # result = result + n
    return result

# numbers is a tuple inside the function:
# total(1, 2)       → numbers = (1, 2)
# total(1,2,3,4,5)  → numbers = (1, 2, 3, 4, 5)

print(total(1, 2))             # 3
print(total(1, 2, 3, 4, 5))   # 15
print(total(10, 20, 30))       # 60


3
15
60


In [19]:
aString = "abcde\n\nfghik\tkkkk"
print(aString)

abcde

fghik	kkkk


In [37]:
seperator = "\n\n"
aStringList = [' Hello How are you?  ',"","Everything alright"]
# for aString in aStringList:
#     tmp = aString.strip()
#     print("original:",f"$${aString}$$", "After striping:",f"$${type(tmp)}$$")

seperator.join(aStringList)

' Hello How are you?  \n\n\n\nEverything alright'

In [38]:
# ShopSmart — combine_context() accepts however many RAG chunks
# the retrieval step returns. Caller never needs to count them.
def combine_context(*chunks: str, separator: str = "\n\n") -> str:
    """
    Join multiple RAG context chunks into one string.
    Empty chunks are silently ignored.
    separator= is keyword-only (comes after *chunks).
    """
    # TRACE: chunks = ('[Doc 1]...', '[Doc 2]...', '')
    # Iter 1 → chunk='[Doc 1]...'  strip() non-empty → include
    # Iter 2 → chunk='[Doc 2]...'  strip() non-empty → include
    # Iter 3 → chunk=''            strip() = '' → skip (falsy)
    # non_empty = ['[Doc 1]...', '[Doc 2]...']
    # result = '[Doc 1]...\n\n[Doc 2]...'
    non_empty = [c.strip() for c in chunks if c.strip()]
    return separator.join(non_empty)

ctx = combine_context(
    "[Doc 1] Classic Monitor: 27-inch 4K display, $205.21.",
    "[Doc 2] Return policy: 30 days from delivery.",
    "",   # empty chunk — ignored automatically
)
print(ctx)
print(f"\nTotal chars: {len(ctx)}")


[Doc 1] Classic Monitor: 27-inch 4K display, $205.21.

[Doc 2] Return policy: 30 days from delivery.

Total chars: 100


## 4 — `**kwargs` — Variable-Length Keyword Arguments

`**kwargs` collects **any number of keyword arguments** into a **dict** inside the function.

```python
def fn(**kwargs):
    # kwargs is a dict of all key=value pairs passed in
    for key, value in kwargs.items():
        ...
```

Use `**kwargs` when callers may pass optional named parameters you cannot predict in advance — like LLM API options.


In [42]:
def aFunction(a,*b,**c):
    print(a)
    print(b)
    print(c)

aFunction(1,2,3,4,5,e=1,f=2,g=3)

1
(2, 3, 4, 5)
{'e': 1, 'f': 2, 'g': 3}


In [46]:
from typing import Any

# ShopSmart — build_api_request() always needs model and messages.
# Everything else (temperature, max_tokens, stream...) is optional
# and changes per call. **kwargs handles all of them.
def build_api_request(model: str, messages: list, **kwargs: Any) -> dict:
    """
    Build an OpenAI-compatible API request payload.
    Required: model, messages.
    Optional via **kwargs: temperature, max_tokens, stream, top_p, etc.
    """
    payload = {"model": model, "messages": messages}
    print(payload)
    print(kwargs)
    payload.update(kwargs)   # merge all optional params
    print(payload)
    return payload

# kwargs = {'temperature': 0.2, 'max_tokens': 512, 'stream': True}
payload = build_api_request(
    model       = "gpt-4o",
    messages    = [{"role": "user", "content": "Hello"}],
    temperature = 0.2,
    max_tokens  = 512,
    stream      = True,
)

for k, v in payload.items():
    if k != "messages":
        print(f"  {k}: {v}")


{'model': 'gpt-4o', 'messages': [{'role': 'user', 'content': 'Hello'}]}
{'temperature': 0.2, 'max_tokens': 512, 'stream': True}
{'model': 'gpt-4o', 'messages': [{'role': 'user', 'content': 'Hello'}], 'temperature': 0.2, 'max_tokens': 512, 'stream': True}
  model: gpt-4o
  temperature: 0.2
  max_tokens: 512
  stream: True


## 5 — Functions as Objects

Functions in Python are **first-class objects** — you can assign them, pass them, and return them.

| Operation | Syntax | Note |
|-----------|--------|------|
| Assign to variable | `op = square` | No `()` — assigns the function itself |
| Call the assigned variable | `op(5)` | Adds `()` to call it |
| Pass into another function | `apply(square, 4)` | The receiving function calls it |

> `square` = the function object  
> `square()` = call the function, get back its return value


In [ ]:
def square(n): return n * n
def cube(n):   return n * n * n

# # Assign function to variable — no () means assign the object, not call it
# operation = square
# print(f"operation = square  →  operation(5) = {operation(5)}")  # 25
#
# operation = cube
# print(f"operation = cube    →  operation(5) = {operation(5)}")  # 125

# Pass a function as an argument
def apply(fn, value):
    """Call fn with value and return the result."""
    return fn(value)

print(f"apply(square, 4) = {apply(square, 4)}")  # 16
print(f"apply(cube,   4) = {apply(cube,   4)}")  # 64


## 6 — lambda · sorted() · map() · filter()

`lambda` is a **one-line anonymous function**. Used almost exclusively with `sorted()`, `map()`, `filter()`.

```python
lambda x: x["price"]      # for each x, return x["price"]
# equivalent to:
def get_price(x):
    return x["price"]
```

| Function | What it does | Returns |
|----------|-------------|--------|
| `sorted(items, key=fn)` | sort by whatever `fn` returns | new sorted list |
| `map(fn, items)` | apply `fn` to every item | iterator (wrap in `list()`) |
| `filter(fn, items)` | keep items where `fn` returns `True` | iterator (wrap in `list()`) |


In [ ]:
prices = [205.21, 568.17, 45.00, 29.99]
print(sorted(prices))

[29.99, 45.0, 205.21, 568.17]


## 7 — List Methods (deep dive)

| Method | What it does | Returns |
|--------|-------------|--------|
| `.append(x)` | Add one item to the end | `None` (modifies in place) |
| `.extend([x,y])` | Add multiple items to the end | `None` |
| `.pop()` | Remove and return last item | the removed item |
| `.pop(i)` | Remove and return item at index i | the removed item |
| `.sort()` | Sort in place ascending | `None` |
| `.sort(reverse=True)` | Sort in place descending | `None` |
| `.reverse()` | Reverse in place | `None` |
| `.index(x)` | First index of value x | `int` |
| `.count(x)` | How many times x appears | `int` |
| `.insert(i, x)` | Insert x at index i | `None` |
| `.remove(x)` | Remove first occurrence of x | `None` |
| `.clear()` | Remove all items | `None` |
| `list[a:b]` | Slice from a to b-1 | new list |


### ShopSmart — LLM message history

The conversation history sent to the OpenAI API is a **list of dicts** that grows one `.append()` at a time.  
This is the exact pattern used in every LLM chat application.


---


In [ ]:
person = {"name": "Alice", "age": 25}

print(".keys()  :", list(person.keys()))
print(".values():", list(person.values()))
print(".items() :", list(person.items()))

print(".get('name')        :", person.get("name"))
print(".get('phone','N/A') :", person.get("phone", "N/A"))  # safe — no KeyError

person.update({"city": "Hyderabad", "age": 26})
print(".update(...):", person)

removed = person.pop("city")
print(".pop('city'):", person, "← removed:", removed)


In [ ]:
# .items() — loop over key-value pairs together
# ITERATION TRACE:
# Iter 1 → key="model",       value="gpt-4o"
# Iter 2 → key="max_tokens",  value=1024
# Iter 3 → key="temperature", value=0.2
# Iter 4 → key="stream",      value=False
model_config = {"model": "gpt-4o", "max_tokens": 1024, "temperature": 0.2, "stream": False}

for key, value in model_config.items():
    print(f"  {key:15s}: {value}")


---


In [ ]:
a = {1, 2, 3, 4}
b = {3, 4, 5, 6}

print(f"a = {a}")
print(f"b = {b}")
print(f"a | b  (union)        : {a | b}")
print(f"a & b  (intersection) : {a & b}")
print(f"a - b  (difference)   : {a - b}")
print(f"a ^ b  (sym diff)     : {a ^ b}")

a.add(10)
print(f"a.add(10)    : {a}")

a.discard(99)   # safe — no error if 99 is not there
print(f"a.discard(99): {a}  ← 99 not in set, no error")

print({1,2}.issubset(a))    # True — {1,2} is contained in a
print(a.issuperset({1,2}))  # True — a contains all of {1,2}


## 10 — Tuple + zip()

### Tuple — ordered, immutable
Use tuples when values **must not change**: config pairs, coordinate pairs, version records, function return values.

### zip() — pair two lists by position
```python
for a, b in zip(list1, list2):
    ...  # a and b are matched by index
```
Stops automatically when the shorter list is exhausted.


In [2]:
aTuple = 1,2
print(aTuple)

(1, 2)


In [ ]:
# Prompt version history — list of tuples
# Tuples prevent accidental modification of the records
prompt_versions = [
    (1, "Initial ShopSmart support prompt"),
    (2, "Added refund handling rules"),
    (3, "Improved output format — Technique 04"),
    (4, "Added few-shot examples — Technique 02"),
]

# ITERATION TRACE:
# Iter 1 → version=1, description="Initial ShopSmart..."
# Iter 2 → version=2, description="Added refund..."
# Iter 3 → version=3, description="Improved output..."
# Iter 4 → version=4, description="Added few-shot..."

for version, description in prompt_versions:
    print(f"  v{version}: {description}")

latest_v, latest_d = prompt_versions[-1]   # unpack last tuple
print(f"\nLatest: v{latest_v} — {latest_d}")


## Summary — Day 03

| Concept | Syntax / Rule |
|---------|---------------|
| `def` | `def fn(p1, p2): ... return result` — write once, call anywhere |
| Default params | `def fn(required, optional=default)` — defaults come AFTER required |
| `*args` | Collects any number of positional args → **tuple** inside fn |
| `**kwargs` | Collects any number of keyword args → **dict** inside fn |
| Functions as objects | `op = square` assigns function · `op(5)` calls it |
| `lambda` | `lambda x: x["price"]` — one-line anonymous function |
| `sorted(items, key=fn)` | Sort by whatever `fn` returns |
| `map(fn, items)` | Apply `fn` to every item → wrap in `list()` |
| `filter(fn, items)` | Keep items where `fn` returns `True` → wrap in `list()` |
| List `.append(x)` | Add one item to end (modifies in place) |
| List `.extend([x,y])` | Add multiple items to end |
| List `.pop()` | Remove and return last item |
| Dict `.get(k, default)` | Safe access — never raises KeyError |
| Dict `.items()` | Loop over key-value pairs together |
| Dict `{**d1, **d2}` | Merge dicts — d2 keys override d1 keys |
| Set `in` | O(1) membership check — instant regardless of size |
| Set `\|` `&` `-` `^` | union · intersection · difference · symmetric diff |
| Tuple unpacking | `a, b = (x, y)` — assign each value in one line |
| `zip(list1, list2)` | Pair items by index · stops at shorter list |

**Connection to Module 00:**
- `build_system_prompt()` → Module 00 Technique 01 prompt anatomy in Python
- `build_api_request()` → assembles the full API payload the LLM receives
- `route_query()` → Module 00 routing rules, now reusable
- `combine_context()` → builds the `<context>` block from Technique 05 (RAG)

**Next:** Day 04 — File I/O + JSON + String Operations  
Loading system prompts from `.txt` files · parsing LLM JSON responses · string methods
